**Author**: Felipe Matheus  
**Start Date**: 29/07/2026  
**End Date**: --/07/2026  
**Purpose**: Experiment launcher ("control panel") for the annealing_iacs surrogate.

This notebook does NOT contain pipeline logic. All the logic (Model A -> OOF ->
NNLS -> Model B -> calibration -> metrics -> persistence) lives in
`src/modeling/Experiments.py`, which orchestrates the existing `Modeling` and
`Evaluation` helpers. Here you only:

1. Load and prepare the data (once).
2. Define a base `ExperimentConfig`.
3. Define the grid of variations you want to sweep.
4. Run and inspect the central `experiments_log.csv`.

Results layout on disk:
```
models/annealing_iacs/experiments/
    experiments_log.csv          <- 1 row per run (the "results spreadsheet")
    <tag>__<hash>/               <- 1 folder per run
        config.yaml
        model_a/   model_b/
        artifacts.pkl
        leaderboard_autogluon.csv
```

# 1. Setup

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TARGET = "elongation_final"
ALL_FEATURES = ["initial_diameter", "tensile_strength", "purity", "elongation", "temperature", "time"]
PROCESS = "annealing_elongation"

SCHEMA_DATE = "110826"
FILE_NAME_SCHEMA_DATA = "schema_annealing_essays_{}.csv".format(SCHEMA_DATE)

TAG = f"annealing-schema{SCHEMA_DATE}"
GRID = {
    "time_limit_a": [900, 600],
    # "num_bag_folds_a": [10, 20],
    # "features": [
    #     ("initial_diameter", "tensile_strength", "purity", "elongation", "temperature", "time"),
    #     ("initial_diameter", "purity", "elongation", "temperature", "time"),
    # ],
}

# 2. Data (same preparation as annealing_elongation.ipynb, run once)

In [3]:
df, df_val = proc.process_annealing_elongation(
    features=ALL_FEATURES,
    target=TARGET,
    df_schema=pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
)

In [4]:
# # SCHEMA_DATE = "080626"
# # FILE_NAME = "dataset_annealing_iacs.csv"
# FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{SCHEMA_DATE}.csv"
# # FILE_NAME_VALIDATION_DATA = "Experimental Results Annealing V2 - CompiledResults.csv"

# # ---- Schema (essay) data: explicit is_essay marker ----
# df_raw_schema = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
# df_schema = df_raw_schema[ALL_FEATURES + [TARGET]].dropna()
# df_schema["is_essay"] = True

# # ---- Literature data ----
# df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
# df_float = proc.df_to_float(df_raw, drop_cols=["DOI"], ignore_columns=["material"])
# df_labeled = feng.label_element(df_float).drop_duplicates()
# df_with_masks = feng.add_ratio_mask_column(
#     feng.add_ratio_mask_column(df_labeled, "grain_size"), "iacs",
# )
# df_lit = df_with_masks[df_with_masks.has_Cu == True][ALL_FEATURES + [TARGET]]
# df_lit["is_essay"] = False

# # ---- Validation data ----
# df_val = proc.load_validation_data_chimie_paris(
#     path=os.path.join(varv.PATHS.data_processed, FILE_NAME_VALIDATION_DATA)
# )

# assert df_lit[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"

# # ---- Concat. weight_col is built INSIDE the runr from is_essay + config ----
# df = pd.concat([df_schema, df_lit], ignore_index=True)
# print(f"Dataset: {df.shape} | essays: {df.is_essay.sum()} | lit: {(~df.is_essay).sum()}")
# df.head()

# 3. Base config

In [ ]:
base = ExperimentConfig(
    process=PROCESS,
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    presets_a = "best_quality",
    #     presets_a = "medium_quality"
    #     presets_b = "medium_quality"
    # everything else uses the defaults; override here if needed, e.g.:
    # time_limit_a=120, weight_on_essay_rows=1.0, use_shared_folds=False,
)
print(base.run_id)

annealing-schema230726__78b06c52


# 4. Single run (sanity check before any grid)

Always run the base config alone first. Then run it 2-3 more times with
`tag="annealing-v1-rep2"` etc. to measure run-to-run noise: AutoGluon under a
time budget is NOT deterministic, and at n~90 this noise is the floor below
which grid differences mean nothing.

In [6]:
result = runr.run_experiment(df, cfg=base, df_val=df_val)
result["artifacts"]["metrics"]

2026-07-29 16:28:09,113 | INFO | src.modeling.Experiments | === Running annealing-schema230726__78b06c52 ===
2026-07-29 16:28:09,114 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       5.36 GB / 31.57 GB (17.0%)
Disk Space Avail:   737.06 GB / 932.08 GB (79.1%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 120s


{'rmse': 8.292139291336353,
 'mae': 5.245794353485107,
 'mape': 25.865454014424767,
 'r2': 0.6084882901723389}

# 5. Validation set

`df_val` rides along inside the runner: after training + calibration, the
calibrated predictive distribution is evaluated on it and `val_rmse`,
`val_mae`, `val_r2`, `val_cov_*` are appended to the SAME log row — so every
grid run carries fold (OOF) metrics AND validation metrics side by side.

Using `df_schema` as `df_val` is illustrative: those rows are inside the
training data, so val metrics are optimistic. Swap in any held-out
DataFrame with the same columns and nothing else changes.

In [7]:
df_val

,initial_diameter,tensile_strength,purity,elongation,temperature,time,elongation_final
9,1.20,431.58,99.9000,0.94,623.0,60.0,26.17
25,2.00,376.08,99.9558,5.33,573.0,30.0,8.55
8,1.20,431.58,99.9000,0.94,623.0,30.0,27.06
21,1.20,335.21,99.9800,1.61,523.0,30.0,51.01
0,2.16,391.69,99.9000,4.90,573.0,30.0,48.60
12,1.20,397.09,99.9000,2.36,573.0,30.0,44.18
17,1.20,397.09,99.9000,2.36,523.0,90.0,42.28
22,1.20,335.21,99.9800,1.61,523.0,60.0,49.42


In [8]:
result["artifacts"]["validation_metrics"]

{'rmse': 10.597993658519528,
 'mae': 6.547539176124967,
 'mape': 48.52385742130462,
 'r2': 0.43481691755998053,
 'coverage': {0.5: 0.25, 0.8: 0.75, 0.9: 0.75, 0.95: 0.875}}

In [9]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [10]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

In [11]:
result

{'cfg': ExperimentConfig(process='annealing_elongation', tag='annealing-schema230726', base_tag=None, target='elongation_final', features=('initial_diameter', 'tensile_strength', 'purity', 'elongation', 'temperature', 'time'), weight_on_essay_rows=1.0, presets_a='medium_quality', num_bag_folds_a=5, num_bag_sets_a=1, num_stack_levels_a=0, time_limit_a=120, presets_b='medium_quality', num_bag_folds_b=5, num_stack_levels_b=0, time_limit_b=60, use_weighted_variance=True, variance_floor_frac=0.01, recalibration_target_alpha=0.9, calibration_alphas=(0.5, 0.8, 0.9, 0.95), y_max=None, y_min=None, fold_seed=42, use_shared_folds=False, group_col=None),
 'run_dir': Path('../../models/annealing_elongation/experiments/annealing-schema230726/annealing-schema230726__78b06c52'),
 'artifacts': {'config': {'process': 'annealing_elongation',
   'tag': 'annealing-schema230726',
   'base_tag': None,
   'target': 'elongation_final',
   'features': ['initial_diameter',
    'tensile_strength',
    'purity',
 

# 6. Grid

Keys are `ExperimentConfig` field names; values are lists of variants.
`features` variants must be tuples. Already-completed runs are skipped
(`force=True` to redo).

In [12]:
log = runr.run_grid(df, base_cfg=base, grid=GRID, df_val=df_val)
log

2026-07-29 16:30:07,520 | INFO | src.modeling.Experiments | Grid: 4 runs over ['time_limit_a', 'features']
2026-07-29 16:30:07,523 | INFO | src.modeling.Experiments | === Running annealing-schema230726__time_limit_a=300__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__5528796e ===
2026-07-29 16:30:07,524 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       4.93 GB / 31.57 GB (15.6%)
Disk Space Avail:   737.05 GB / 932.08 GB (79.1%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample w

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,mean_sigma_aleat,n_val,val_rmse,val_mae,val_mape,val_r2,val_cov_0.5,val_cov_0.8,val_cov_0.9,val_cov_0.95
0,annealing-schema230726__78b06c52,2026-07-29T16:30:05,116.4,28,e85dd375,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_elongation\experiments\annealing-schema230726\annealing-schema230726__78b06c52,annealing_elongation,annealing-schema230726,NaN,elongation_final,...,1.34956,8,10.59799,6.54754,48.52386,0.43482,0.25,0.75,0.750,0.875
1,annealing-schema230726__time_limit_a=300__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__5528796e,2026-07-29T16:31:54,107.2,28,e85dd375,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_elongation\experiments\annealing-schema230726\annealing-schema230726__time_limit_a=300__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__5528796e,annealing_elongation,annealing-schema230726__time_limit_a=300__features=initial_diameter-tensile_strength-purity-elongation-temperature-time,annealing-schema230726,elongation_final,...,1.34956,8,10.59799,6.54754,48.52386,0.43482,0.25,0.75,0.750,0.875
2,annealing-schema230726__time_limit_a=300__features=initial_diameter-purity-elongation-temperature-time__a850a6c4,2026-07-29T16:33:44,109.4,28,b52f1456,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_elongation\experiments\annealing-schema230726\annealing-schema230726__time_limit_a=300__features=initial_diameter-purity-elongation-temperature-time__a850a6c4,annealing_elongation,annealing-schema230726__time_limit_a=300__features=initial_diameter-purity-elongation-temperature-time,annealing-schema230726,elongation_final,...,1.34956,8,9.34959,5.60851,42.38869,0.56013,0.25,0.75,0.875,0.875
3,annealing-schema230726__time_limit_a=600__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__1bdeffd2,2026-07-29T16:35:29,105.6,28,e85dd375,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_elongation\experiments\annealing-schema230726\annealing-schema230726__time_limit_a=600__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__1bdeffd2,annealing_elongation,annealing-schema230726__time_limit_a=600__features=initial_diameter-tensile_strength-purity-elongation-temperature-time,annealing-schema230726,elongation_final,...,1.34956,8,10.59799,6.54754,48.52386,0.43482,0.25,0.75,0.750,0.875
4,annealing-schema230726__time_limit_a=600__features=initial_diameter-purity-elongation-temperature-time__1718d052,2026-07-29T16:37:10,100.5,28,b52f1456,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_elongation\experiments\annealing-schema230726\annealing-schema230726__time_limit_a=600__features=initial_diameter-purity-elongation-temperature-time__1718d052,annealing_elongation,annealing-schema230726__time_limit_a=600__features=initial_diameter-purity-elongation-temperature-time,annealing-schema230726,elongation_final,...,1.34956,8,9.34959,5.60851,42.38869,0.56013,0.25,0.75,0.875,0.875


# 7. Inspect results

In [14]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_weight_on_essay_rows", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9", "c_opt", "pct_truncated_aleat",
    "mean_sigma_epist", "mean_sigma_aleat", "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_weight_on_essay_rows,cfg_features,rmse,mae,cov_0.9,val_rmse,val_mae,val_cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
2,annealing-schema230726__time_limit_a=300__features=initial_diameter-purity-elongation-temperature-time__a850a6c4,300,1.0,initial_diameter|purity|elongation|temperature|time,8.00787,4.94571,0.8929,9.34959,5.60851,0.875,2.0930,25.0,2.29640,1.34956,109.4
4,annealing-schema230726__time_limit_a=600__features=initial_diameter-purity-elongation-temperature-time__1718d052,600,1.0,initial_diameter|purity|elongation|temperature|time,8.00787,4.94571,0.8929,9.34959,5.60851,0.875,2.0930,25.0,2.29640,1.34956,100.5
1,annealing-schema230726__time_limit_a=300__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__5528796e,300,1.0,initial_diameter|tensile_strength|purity|elongation|temperature|time,8.29214,5.24579,0.8571,10.59799,6.54754,0.750,2.4098,25.0,1.85944,1.34956,107.2
0,annealing-schema230726__78b06c52,120,1.0,initial_diameter|tensile_strength|purity|elongation|temperature|time,8.29214,5.24579,0.8571,10.59799,6.54754,0.750,2.4098,25.0,1.85944,1.34956,116.4
3,annealing-schema230726__time_limit_a=600__features=initial_diameter-tensile_strength-purity-elongation-temperature-time__1bdeffd2,600,1.0,initial_diameter|tensile_strength|purity|elongation|temperature|time,8.29214,5.24579,0.8571,10.59799,6.54754,0.750,2.4098,25.0,1.85944,1.34956,105.6


In [15]:
# Quick pivot: effect of one knob, marginalised over the others.
# Remember: compare against run-to-run noise (Section 4) before concluding.
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                mae           cov_0.9          
                      mean       std     mean       std    mean       std
cfg_time_limit_a                                                         
120               8.292140       NaN  5.24579       NaN  0.8571       NaN
300               8.150005  0.201009  5.09575  0.212189  0.8750  0.025314
600               8.150005  0.201009  5.09575  0.212189  0.8750  0.025314

# 8. Load a winner for deployment / further analysis

Each run folder is self-contained: predictors + artifacts.pkl with weights,
`recalibration_c`, calibration tables.

In [16]:
# log already carries the resolved absolute path, so use it directly
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-schema230726__time_limit_a=300__features=initial_diameter-purity-elongation-temperature-time__a850a6c4


,alpha,empirical_coverage,gap
0,0.50,0.607143,0.107143
1,0.80,0.857143,0.057143
2,0.90,0.892857,-0.007143
3,0.95,0.964286,0.014286
